# Fish Weight Prediction (Regression)

This notebook trains regression models to predict **Weight** from fish measurements in `Fish.csv`.

**Dataset:** `Fish.csv` (uploaded by user)  
**Target:** `Weight`  
**Features:** `Length1, Length2, Length3, Height, Width`  


In [ ]:
# --- Setup ---
import pandas as pd
import numpy as np

import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split, KFold, cross_validate
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import StandardScaler
from sklearn.impute import SimpleImputer

from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

from sklearn.linear_model import LinearRegression, Ridge, Lasso, ElasticNet
from sklearn.neighbors import KNeighborsRegressor
from sklearn.svm import SVR
from sklearn.tree import DecisionTreeRegressor
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor

# For heteroscedasticity test (optional)
import statsmodels.api as sm
from statsmodels.stats.diagnostic import het_breuschpagan

DATA_PATH = "Fish.csv"
df = pd.read_csv(DATA_PATH)

print("Shape:", df.shape)
display(df.head())


## 1) Quick EDA

In [ ]:
# Basic info
display(df.describe(include="all"))

# Check missing values
print("Missing values per column:")
print(df.isna().sum())

# Target distribution
plt.figure()
plt.hist(df["Weight"], bins=20)
plt.title("Weight Distribution")
plt.xlabel("Weight")
plt.ylabel("Count")
plt.show()

# Correlation with Weight (numeric only)
num_cols = df.select_dtypes(include=[np.number]).columns.tolist()
corr = df[num_cols].corr(numeric_only=True)["Weight"].sort_values(ascending=False)
print("Correlation with Weight:")
display(corr)

# Simple scatter plots vs target
features = [c for c in ["Length1","Length2","Length3","Height","Width"] if c in df.columns]
for c in features:
    plt.figure()
    plt.scatter(df[c], df["Weight"], alpha=0.7)
    plt.title(f"Weight vs {c}")
    plt.xlabel(c)
    plt.ylabel("Weight")
    plt.show()


## 2) Train/Test Split + Preprocessing

We will:
- Split into train/test
- Use a preprocessing pipeline:
  - Impute missing values (median)
  - Scale numeric features (StandardScaler) for models that benefit (e.g., SVR, KNN, linear models)


In [ ]:
target = "Weight"

# Define features
feature_cols = [c for c in ["Length1","Length2","Length3","Height","Width","Species"] if c in df.columns and c != target]
X = df[feature_cols].copy()
y = df[target].copy()

# If Species exists, we will drop it for pure numeric regression
# (You can one-hot encode it if you want; here we keep things simple for a classic regression example.)
if "Species" in X.columns:
    print("Dropping 'Species' for this regression baseline (optional).")
    X = X.drop(columns=["Species"])

numeric_features = X.columns.tolist()

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

print("Train:", X_train.shape, "Test:", X_test.shape)

numeric_transformer = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler())
])

preprocess = ColumnTransformer(
    transformers=[("num", numeric_transformer, numeric_features)],
    remainder="drop"
)


## 3) Model Benchmarks (Cross-Validation)

We will compare several regression models using 5-fold CV on the training set.

Metrics reported:
- **MAE** (lower is better)
- **RMSE** (lower is better)
- **R²** (higher is better)


In [ ]:
models = {
    "LinearRegression": LinearRegression(),
    "Ridge(alpha=1.0)": Ridge(alpha=1.0, random_state=42),
    "Lasso(alpha=0.001)": Lasso(alpha=0.001, random_state=42, max_iter=10000),
    "ElasticNet(alpha=0.001,l1=0.5)": ElasticNet(alpha=0.001, l1_ratio=0.5, random_state=42, max_iter=10000),
    "KNN(k=5)": KNeighborsRegressor(n_neighbors=5),
    "SVR(RBF,C=10,eps=0.1)": SVR(C=10, epsilon=0.1, kernel="rbf"),
    "DecisionTree": DecisionTreeRegressor(random_state=42),
    "RandomForest(n=300)": RandomForestRegressor(n_estimators=300, random_state=42),
    "GradientBoosting": GradientBoostingRegressor(random_state=42),
}

def rmse(y_true, y_pred):
    return np.sqrt(mean_squared_error(y_true, y_pred))

scoring = {
    "MAE": "neg_mean_absolute_error",
    "RMSE": "neg_root_mean_squared_error",
    "R2": "r2",
}

cv = KFold(n_splits=5, shuffle=True, random_state=42)

rows = []
for name, model in models.items():
    pipe = Pipeline(steps=[("preprocess", preprocess), ("model", model)])
    cv_res = cross_validate(pipe, X_train, y_train, cv=cv, scoring=scoring, n_jobs=-1)
    rows.append({
        "model": name,
        "MAE (CV)": -cv_res["test_MAE"].mean(),
        "RMSE (CV)": -cv_res["test_RMSE"].mean(),
        "R2 (CV)": cv_res["test_R2"].mean(),
    })

results = pd.DataFrame(rows).sort_values(by="RMSE (CV)")
display(results)


## 4) Train Best Model on Full Training Set + Evaluate on Test Set

Pick the best model from the table above (lowest CV RMSE) and evaluate on the held-out test set.


In [ ]:
# Choose best model name from results (lowest CV RMSE)
best_name = results.iloc[0]["model"]
print("Best by CV RMSE:", best_name)

best_model = models[best_name]

best_pipe = Pipeline(steps=[("preprocess", preprocess), ("model", best_model)])
best_pipe.fit(X_train, y_train)

y_pred = best_pipe.predict(X_test)

mae = mean_absolute_error(y_test, y_pred)
rmse_val = rmse(y_test, y_pred)
r2 = r2_score(y_test, y_pred)

print(f"Test MAE:  {mae:.3f}")
print(f"Test RMSE: {rmse_val:.3f}")
print(f"Test R2:   {r2:.3f}")


## 5) Diagnostics: Residuals, Predicted vs Actual

These plots help you check:
- Fit quality
- Outliers
- Non-linearity patterns
- Heteroscedasticity (variance changing with prediction)


In [ ]:
residuals = y_test - y_pred

# Predicted vs Actual
plt.figure()
plt.scatter(y_test, y_pred, alpha=0.8)
plt.title("Predicted vs Actual (Test Set)")
plt.xlabel("Actual Weight")
plt.ylabel("Predicted Weight")
plt.plot([y_test.min(), y_test.max()], [y_test.min(), y_test.max()])  # y=x
plt.show()

# Residual plot
plt.figure()
plt.scatter(y_pred, residuals, alpha=0.8)
plt.title("Residuals vs Predicted (Test Set)")
plt.xlabel("Predicted Weight")
plt.ylabel("Residual (Actual - Predicted)")
plt.axhline(0)
plt.show()

# Residual distribution
plt.figure()
plt.hist(residuals, bins=20)
plt.title("Residual Distribution (Test Set)")
plt.xlabel("Residual")
plt.ylabel("Count")
plt.show()


## 6) Optional: Breusch–Pagan Test for Heteroscedasticity

**Null hypothesis:** residual variance is constant (homoscedastic).  
If p-value is small (e.g., < 0.05), it suggests heteroscedasticity.

> Note: This test assumes a linear-type model structure. It's still a useful diagnostic even if your final model is non-linear.


In [ ]:
# Build design matrix for BP test using the preprocessed features
X_train_proc = preprocess.fit_transform(X_train)  # fit on train
X_test_proc = preprocess.transform(X_test)

# Add constant for statsmodels
X_bp = sm.add_constant(X_test_proc)
bp = het_breuschpagan(residuals, X_bp)

labels = ["LM stat", "LM p-value", "F stat", "F p-value"]
print(dict(zip(labels, bp)))


## 7) Save the Trained Model (Optional)

This saves the trained pipeline to disk so you can load it later for predictions.


In [ ]:
import joblib

MODEL_PATH = "fish_weight_model.joblib"
joblib.dump(best_pipe, MODEL_PATH)
print("Saved:", MODEL_PATH)

# Example: load and predict
loaded = joblib.load(MODEL_PATH)
example = X_test.iloc[:5].copy()
print("Example input:")
display(example)
print("Predictions:", loaded.predict(example))
